# RAG (file_search) test

**QE Perspective:** We validate end-to-end RAG: create a vector store, upload a document, ingest it, then call the Responses API with `file_search` and assert the answer is grounded in the document. We also check **negative** (invalid vector_store_id yields an error) and **edge** (missing config, no vector_io provider fail fast). This ensures the RAG pipeline and API contract are stable.

- **Positive:** Vector store + file upload + file_search → response contains expected fact (e.g. "1980").
- **Negative:** Request with non-existent vector_store_id → API returns error.
- **Edge:** Assert base_url/model set; assert at least one vector_io provider before running.

Aligned with [llama-stack-demos simple_rag](https://github.com/opendatahub-io/llama-stack-demos/blob/main/demos/03_rag/01_simple_rag.py). Config: `BASE_URL`, `MODEL` (optional: `EMBEDDING_MODEL`, `EMBEDDING_DIMENSION`). Run via pytest or interactively.


## Setup

Load config from env; optionally import `response_text` from `scripts.helpers` for consistent response parsing. Create client and ensure vector_io provider exists.


In [ ]:
import os
from scripts.helpers import response_text


base_url = os.environ.get("BASE_URL", "http://localhost:8321")
model = os.environ.get("MODEL", "vllm-inference/llama-3-2-3b")
embedding_model = os.environ.get(
    "EMBEDDING_MODEL", "vllm-embedding/ibm-granite/granite-embedding-125m-english"
)
embedding_dimension = int(os.environ.get("EMBEDDING_DIMENSION", "1536"))

# Edge: fail fast if config missing
assert base_url, "BASE_URL must be set (e.g. http://localhost:8321)"
assert model, "MODEL must be set for RAG inference"

In [ ]:
from llama_stack_client import LlamaStackClient

# Client expects base URL without /v1 (it appends /v1/ to paths)
client = LlamaStackClient(base_url=base_url)

assert model, "MODEL must be set for RAG inference"
assert embedding_model, "EMBEDDING_MODEL must be set for RAG inference"

# Check for model
client.models.list()

print(client.models.list())

# if model or embedding_model is not found, raise an error
# [Model(id='sentence-transformers/ibm-granite/granite-embedding-125m-english', created=1772046830, owned_by='llama_stack', custom_metadata={'model_type': 'embedding', 'provider_id': 'sentence-transformers', 'provider_resource_id': 'ibm-granite/granite-embedding-125m-english', 'embedding_dimension': 768}, object='model'), Model(id='sentence-transformers/nomic-ai/nomic-embed-text-v1.5', created=1772046830, owned_by='llama_stack', custom_metadata={'model_type': 'embedding', 'provider_id': 'sentence-transformers', 'provider_resource_id': 'nomic-ai/nomic-embed-text-v1.5', 'embedding_dimension': 768}, object='model'), Model(id='vllm-embedding/ibm-granite/granite-embedding-125m-english', created=1772046830, owned_by='llama_stack', custom_metadata={'model_type': 'embedding', 'provider_id': 'vllm-embedding', 'provider_resource_id': 'ibm-granite/granite-embedding-125m-english', 'embedding_dimension': 768}, object='model'), Model(id='vllm-inference/llama-3-2-3b', created=1772046830, owned_by='llama_stack', custom_metadata={'model_type': 'llm', 'provider_id': 'vllm-inference', 'provider_resource_id': 'llama-3-2-3b'}, object='model')]

if model not in [m.id for m in client.models.list()]:
    raise ValueError(f"Model {model} not found")
if embedding_model not in [m.id for m in client.models.list()]:
    raise ValueError(f"Model {embedding_model} not found")

INFO:httpx:HTTP Request: GET http://localhost:8321/v1/models "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8321/v1/models "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8321/v1/models "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8321/v1/models "HTTP/1.1 200 OK"


[Model(id='sentence-transformers/ibm-granite/granite-embedding-125m-english', created=1772046927, owned_by='llama_stack', custom_metadata={'model_type': 'embedding', 'provider_id': 'sentence-transformers', 'provider_resource_id': 'ibm-granite/granite-embedding-125m-english', 'embedding_dimension': 768}, object='model'), Model(id='sentence-transformers/nomic-ai/nomic-embed-text-v1.5', created=1772046927, owned_by='llama_stack', custom_metadata={'model_type': 'embedding', 'provider_id': 'sentence-transformers', 'provider_resource_id': 'nomic-ai/nomic-embed-text-v1.5', 'embedding_dimension': 768}, object='model'), Model(id='vllm-embedding/ibm-granite/granite-embedding-125m-english', created=1772046927, owned_by='llama_stack', custom_metadata={'model_type': 'embedding', 'provider_id': 'vllm-embedding', 'provider_resource_id': 'ibm-granite/granite-embedding-125m-english', 'embedding_dimension': 768}, object='model'), Model(id='vllm-inference/llama-3-2-3b', created=1772046927, owned_by='llam

In [26]:
# Edge: no vector_io provider → cannot run RAG
vector_providers = [p for p in client.providers.list() if p.api == "vector_io"]
assert vector_providers, "No vector_io provider available; RAG test cannot run"
selected_vector_provider = vector_providers[0]

INFO:httpx:HTTP Request: GET http://localhost:8321/v1/providers "HTTP/1.1 200 OK"


In [ ]:
# Positive: create vector store (demo-style extra_body), upload doc, file_search, assert answer
from io import BytesIO
from uuid import uuid4

doc_text = """Bering Land Bridge National Preserve. Proclaimed a national monument Dec. 1, 1978; established as a national preserve Dec. 2, 1980.
Denali National Park. Established as Mt. McKinley National Park Feb. 26, 1917. Designated Denali National Park and Preserve Dec. 2, 1980."""
question = "When was Bering Land Bridge established as a national preserve?"

vector_store = None
uploaded_file = None
try:
    vector_store = client.vector_stores.create(
        name=f"rag_test_{uuid4().hex[:8]}",
        extra_body={
            "provider_id": selected_vector_provider.provider_id,
            "embedding_model": embedding_model,
            "embedding_dimension": embedding_dimension,
        },
    )
    file_buffer = BytesIO(doc_text.encode("utf-8"))
    file_buffer.name = "rag_doc.txt"
    uploaded_file = client.files.create(file=file_buffer, purpose="assistants")
    client.vector_stores.files.create(
        vector_store_id=vector_store.id,
        file_id=uploaded_file.id,
        chunking_strategy={
            "type": "static",
            "static": {"max_chunk_size_tokens": 256, "chunk_overlap_tokens": 32},
        },
    )
    response = client.responses.create(
        model=model,
        instructions="Use file_search to answer the question using the provided documents.",
        input=[{"role": "user", "content": question}],
        tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
        tool_choice={"type": "file_search"},
        stream=False,
    )
    assert (
        response.status == "completed"
    ), f"Expected status completed, got {response.status}"
    text = response_text(response)
    assert "1980" in text, f"Expected '1980' in answer from doc, got: {text[:300]}"
finally:
    if vector_store:
        try:
            client.vector_stores.delete(vector_store_id=vector_store.id)
        except Exception:
            pass
    if uploaded_file:
        try:
            client.files.delete(file_id=uploaded_file.id)
        except Exception:
            pass

INFO:httpx:HTTP Request: POST http://localhost:8321/v1/vector_stores "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8321/v1/files "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8321/v1/vector_stores/vs_8e35a92d-6465-4de2-bfdd-f9390999dd20/files "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8321/v1/responses "HTTP/1.1 500 Internal Server Error"
INFO:llama_stack_client._base_client:Retrying request to /v1/vector_stores/vs_8e35a92d-6465-4de2-bfdd-f9390999dd20 in 0.446203 seconds
INFO:httpx:HTTP Request: DELETE http://localhost:8321/v1/vector_stores/vs_8e35a92d-6465-4de2-bfdd-f9390999dd20 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: DELETE http://localhost:8321/v1/files/file-ba25ac35082243fdbd53616b94b84522 "HTTP/1.1 200 OK"


InternalServerError: Error code: 500 - {'error': {'detail': 'Internal server error: An unexpected error occurred.'}}

In [ ]:
# Negative: file_search with invalid vector_store_id should fail
raised = False
try:
    client.responses.create(
        model=model,
        input=[{"role": "user", "content": "What is 2+2?"}],
        tools=[{"type": "file_search", "vector_store_ids": ["vs_nonexistent_invalid"]}],
        tool_choice={"type": "file_search"},
        stream=False,
    )
except Exception as e:
    raised = True
assert raised, "Expected an error when using invalid vector_store_id"

In [ ]:
# RAG test done: positive (file_search with real store), negative (invalid store id), edge (config + no provider)